## Embedding Model Training on Google Colab

In [ ]:
### Library Installation in Colab

import os
if "COLAB_" not in "".join(os.environ.keys()):
    !pip install unsloth
else:
    # Do this only in Colab notebooks! Otherwise use pip install unsloth
    !pip install --no-deps bitsandbytes accelerate xformers==0.0.29.post3 peft trl==0.15.2 triton cut_cross_entropy unsloth_zoo
    !pip install sentencepiece protobuf "datasets>=3.4.1" huggingface_hub hf_transfer
    !pip install transformers==4.51.3
    !pip install --no-deps unsloth
!pip install sentence_transformers bert-score nltk rouge-score scikit-learn cmaes scipy
!pip install evaluate
!pip install sacrebleu
!pip install llama_index optuna optunahub
!pip install rapidfuzz
!pip install git+https://github.com/brandonstarxel/chunking_evaluation.git
!pip install -U langchain-community
!pip install llama-index-embeddings-langchain cohere langchain-openai langdetect langgraph

In [28]:
import pandas as pd
import numpy as np
import datasets
import torch
from sentence_transformers.evaluation import InformationRetrievalEvaluator, SentenceEvaluator, SequentialEvaluator
from sentence_transformers.util import cos_sim
from sentence_transformers.losses import MatryoshkaLoss,MultipleNegativesRankingLoss
from sentence_transformers import SentenceTransformer, SentenceTransformerModelCardData, SentenceTransformerTrainingArguments, SentenceTransformerTrainer
from sentence_transformers.training_args import BatchSamplers
from transformers import EarlyStoppingCallback
from huggingface_hub import create_repo, upload_folder
from typing import List
import sys, os
sys.path.append(os.path.abspath("../../../"))
from backend.database.config.config import settings

In [ ]:
### Only in Colab
from google.colab import drive, files
drive.mount('/content/drive')
path = '/content/drive/MyDrive/AILA_project/files_and_data'

### Dataset Creation

In [ ]:
p_folder = f'{os.getcwd()}'.split('\\')[:-1]
path_folder = '//'.join(p for p in p_folder)+'//'
print(path_folder)

df = pd.read_csv(f'{path_folder}/synthetic_dataset_new/more_queries.csv')

ids = np.unique(df['corpus_id'].to_list())
chunks_ids = {ids[i]:i for i in range(len(ids))}

texts = {}
for i in range(len(ids)):
    new_path = path_folder + '/' + ids[i].split('//')[-2] + '/' + ids[i].split('//')[-1]
    with open(new_path,'r',encoding='utf-8') as f: text = f.read()
    texts[ids[i]] = [new_path,text]

data = []
for i in range(len(df)):
    id = chunks_ids[df['corpus_id'][i]]
    text = texts[df['corpus_id'][i]][1]
    data.append([df['question'][i],text,id,i])

df = pd.DataFrame(data,columns=['anchor','positive','global_chunk_id','id'])
df.to_csv(f'{path_folder}/synthetic_dataset_new/new_queries.csv')

c://Users//johnk//Documents//GitHub//AILA-application//backend//evaluation//


In [14]:
dataset = datasets.Dataset.from_pandas(df)
dataset = dataset.shuffle(seed=42)

dataset = dataset.train_test_split(test_size=0.2)
train_dataset = dataset['train']
test_dataset = dataset['test']

corpus_dataset =  datasets.concatenate_datasets([train_dataset,test_dataset])
corpus =  dict(zip(corpus_dataset['id'],corpus_dataset['positive']))
queries = dict(zip(test_dataset['id'],test_dataset['anchor']))

relevant_docs = {}

for q_id,global_chunk_id in zip(test_dataset['id'],test_dataset['global_chunk_id']):
    if q_id not in relevant_docs: relevant_docs[q_id] = []
    matching_corpus_ids = [cid for cid,chunk in zip(corpus_dataset['id'],corpus_dataset['global_chunk_id']) if chunk == global_chunk_id]
    relevant_docs[q_id].extend(matching_corpus_ids)


{3: [4, 2, 1, 3, 0], 41: [34, 36, 46, 55, 52, 40, 39, 49, 38, 56, 42, 33, 48, 35, 45, 37, 53, 44, 51, 54, 41, 50, 58, 43, 57, 47], 75: [76, 79, 80, 78, 77, 75], 30: [26, 32, 24, 31, 28, 25, 29, 21, 30, 23, 27, 22], 102: [101, 99, 100, 102], 23: [26, 32, 24, 31, 28, 25, 29, 21, 30, 23, 27, 22], 14: [19, 15, 13, 17, 18, 8, 11, 5, 9, 6, 20, 7, 10, 16, 14, 12], 27: [26, 32, 24, 31, 28, 25, 29, 21, 30, 23, 27, 22], 50: [34, 36, 46, 55, 52, 40, 39, 49, 38, 56, 42, 33, 48, 35, 45, 37, 53, 44, 51, 54, 41, 50, 58, 43, 57, 47], 22: [26, 32, 24, 31, 28, 25, 29, 21, 30, 23, 27, 22], 83: [84, 81, 82, 83], 58: [34, 36, 46, 55, 52, 40, 39, 49, 38, 56, 42, 33, 48, 35, 45, 37, 53, 44, 51, 54, 41, 50, 58, 43, 57, 47], 43: [34, 36, 46, 55, 52, 40, 39, 49, 38, 56, 42, 33, 48, 35, 45, 37, 53, 44, 51, 54, 41, 50, 58, 43, 57, 47], 92: [93, 92], 12: [19, 15, 13, 17, 18, 8, 11, 5, 9, 6, 20, 7, 10, 16, 14, 12], 57: [34, 36, 46, 55, 52, 40, 39, 49, 38, 56, 42, 33, 48, 35, 45, 37, 53, 44, 51, 54, 41, 50, 58, 43, 

### Define Matryoshka Evaluator

In [15]:
embedding_model_dimensions = {
    "multilingual-e5-large": [1024, 768, 512, 256, 128, 64],
    "legal-bert-base-uncased": [768, 512, 256, 128, 64],
    "modernbert-embed-base": [768, 512, 256, 128, 64],
    "bge-m3": [768, 512, 256, 128, 64], 
    "all-MiniLM-L6-v2":[768, 512, 256, 128, 64], 
    "all-mpnet-base-v2": [512, 256, 128, 64],
    "bert-base-uncased":[768, 512, 256, 128, 64], 
    "distilbert-base-uncased": [768, 512, 256, 128, 64],   
}

In [30]:
def matryoshka_evaluator_model(model_dimensions:List[int],queries:dict,corpus:dict,relevant_docs:dict) -> SentenceEvaluator:
    matryoshka_evaluators = []
    for dim in model_dimensions:
        ir_evaluator = InformationRetrievalEvaluator(
            queries = queries,
            corpus = corpus,
            relevant_docs = relevant_docs,
            name = f'dim_{dim}',
            truncate_dim= dim,
            score_functions={'cosine':cos_sim}
        )

        matryoshka_evaluators.append(ir_evaluator)

    evaluator = SequentialEvaluator(matryoshka_evaluators)
    return evaluator

### Display Results

In [ ]:
def display_results(evaluator: SentenceTransformer, model: SentenceTransformer, model_dimensions:List[int], model_name:str, finetuning:bool):
    base_results = evaluator(model)

    if finetuning: type = 'Finetuned'
    else: type = 'Base'

    print(type)
    print(f"\n {model_name} {type} Model Evaluation Results")
    print("-" * 85)
    string = f"{'Metric':15} " + ' '.join(f'{dim:>12}' for dim in model_dimensions)
    print(string)
    print("-" * 85)

    # List of metrics to display
    metrics = [
        'ndcg@10',
        'mrr@10',
        'map@100',
        'accuracy@1',
        'accuracy@3',
        'accuracy@5',
        'accuracy@10',
        'precision@1',
        'precision@3',
        'precision@5',
        'precision@10',
        'recall@1',
        'recall@3',
        'recall@5',
        'recall@10'
    ]

    for metric in metrics:
        values = []
        for dim in model_dimensions:
            key = f'dim_{dim}_cosine_{metric}'
            values.append(base_results[key])

        metric_name = f"=={metric}==" if metric == 'ndcg@10' else metric
        print(f"{metric_name:15}",end="  ")
        for val in values:
            print(f"{val:12.4f}",end="  ")
        print()

    print("-" * 85)
    print(f"{'seq_score:'} {base_results['sequential_score']:1f}")

### Finetuning Embedding Model

In [ ]:
def finetune_embedding_model(model:SentenceTransformer,
                             model_name:str,
                             num_train_epochs:int,
                             train_batch_size:int,
                             gradient_step:int,
                             eval_batch_size:int,
                             lr:float,
                             evaluator:SentenceEvaluator,
                             train_loss: MatryoshkaLoss):

    args = SentenceTransformerTrainingArguments(
        output_dir = model_name,
        num_train_epochs=num_train_epochs,                                        # number of epochs
        per_device_train_batch_size=train_batch_size,                            # train batch size
        gradient_accumulation_steps=gradient_step,                            # for a global batch size of 512
        per_device_eval_batch_size=eval_batch_size,                             # evaluation batch size
        warmup_ratio=0.1,                                          # warmup ratio
        learning_rate=lr,                                        # learning rate, 2e-5 is a good value
        lr_scheduler_type="cosine",                                # use cosine learning rate scheduler
        optim="adamw_torch_fused",                                 # use fused adamw optimizer
        tf32=True,                                                 # use tf32 precision
        bf16=True,                                                 # use bf16 precision
        batch_sampler=BatchSamplers.NO_DUPLICATES,                 # MultipleNegativesRankingLoss benefits from no duplicate samples in a batch
        eval_strategy="epoch",                                     # evaluate after each epoch
        save_strategy="epoch",                                     # save after each epoch
        logging_steps=1,                                          # log every 10 steps
        save_total_limit=3,                                        # save only the last 3 models
        load_best_model_at_end=True,                               # load the best model when training ends
        metric_for_best_model="eval_dim_128_cosine_ndcg@10",       # Optimizing for the best ndcg@10 score for the 128 dimension
        report_to="none"
    )

    trainer = SentenceTransformerTrainer(
        model = model,
        args = args,
        train_dataset=train_dataset.select_columns(['anchor','positive']),
        loss = train_loss,
        evaluator = evaluator,
        callbacks=[EarlyStoppingCallback(early_stopping_patience=3)]
    )

    trainer.train()

    trainer.save_model()
    repo_basename = model_name
    repo_id = f"IoannisKat1/{repo_basename}-new"
    create_repo(repo_id, private=False, exist_ok=True, token=settings.HF_TOKEN)
    upload_folder(repo_id=repo_id, folder_path=args.output_dir, commit_message="Add finetuned model", token=settings.HF_TOKEN)

### Pulling it all together

In [ ]:
model_name = 'intfloat/multilingual-e5-large'

if model_name.split('/')[-1] not in embedding_model_dimensions.keys(): raise RuntimeError("Model not in the available models for finetuning")

model = SentenceTransformer(model_name,device='cuda' if torch.cuda.is_available() else 'cpu')
model_name_ = model_name.split('/')[1]

evaluator = matryoshka_evaluator_model(embedding_model_dimensions[model_name_],queries,corpus,relevant_docs)

display_results(evaluator,model,embedding_model_dimensions[model_name_],model_name_,False)

model = SentenceTransformer(
    model_name_or_path = model_name,
    model_kwargs= {'attn_implementation':'sdpa'},
    model_card_data=SentenceTransformerModelCardData(
        language='en',
        license='apache-2.0',
        model_name = f"{model_name_.replace('-','_')} Finetuned on Data"
    )
)

base_loss = MultipleNegativesRankingLoss(model)

train_loss = MatryoshkaLoss(model, base_loss,matryoshka_dims=embedding_model_dimensions[model_name_])

finetune_embedding_model(model,model_name_,num_train_epochs=10,train_batch_size=8,gradient_step=2,eval_batch_size=8,lr=2e-5,evaluator=evaluator,train_loss=train_loss)

display_results(evaluator,model,embedding_model_dimensions[model_name_],model_name_,True)

YES


KeyboardInterrupt: 